# Notebook 1 - Data Collection and Infrastructure

**Project:** Macro Divergence and Sovereign Bond Value: A Cross-Country Investment Framework  
**Author:** Thomas Sumner  
**Date:** May 2026  

---

## Overview

This notebook pulls and cleans all raw data required across the project. Data is sourced from 
FRED, Bank of England, ONS, ECB Statistical Data Warehouse, IMF World Economic Outlook, and 
yfinance. All processed CSVs are saved to `/data/processed/` and reusable loader functions 
are written to `src/data_loader.py` for use across all subsequent notebooks.

**Economies covered:** United States, United Kingdom, Eurozone (Germany as sovereign benchmark), Japan  
**Sample period:** January 2015 – May 2026  
**Data as of:** 1 May 2026  

---

## Data Sources

| Source | Contents |
|--------|----------|
| FRED | US/EUR/JPN yields, TIPS, breakevens, macro series, FX, credit spreads, commodities |
| Bank of England | UK nominal, real and inflation gilt spot curves |
| ONS | UK CPI (headline, core, services), unemployment, GDP |
| ECB SDW | Eurozone HICP decomposition, AAA yield curve, unemployment |
| IMF WEO | Output gap estimates — all four economies, April 2026 vintage |
| yfinance | Equity indices, gold |

---

## Processed CSV Outputs

| File | Contents | Frequency |
|------|----------|-----------|
| `yields_daily.csv` | US Treasury curve, EUR Bund 10Y, JPN JGB 10Y | Daily |
| `real_yields_daily.csv` | US TIPS real yields, 10Y breakeven | Daily |
| `fx_daily.csv` | GBPUSD, EURUSD, USDJPY | Daily |
| `credit_daily.csv` | US IG OAS, US HY OAS, EUR HY OAS (Apr 2023 onwards) | Daily |
| `market_daily.csv` | VIX, Brent, Fed Funds daily, ECB rate daily | Daily |
| `boe_nominal_daily.csv` | UK nominal gilt spot curve — 2Y, 5Y, 10Y, 30Y | Daily |
| `boe_real_daily.csv` | UK real gilt spot curve — 5Y, 10Y, 30Y | Daily |
| `boe_inflation_daily.csv` | UK inflation breakeven curve — 5Y, 10Y, 30Y | Daily |
| `ecb_aaa_yields_daily.csv` | ECB AAA yield curve — 2Y, 5Y, 10Y, 30Y | Daily |
| `equities_commodities_daily.csv` | S&P 500, FTSE 100, Euro Stoxx 50, Nikkei, Gold, Brent | Daily |
| `macro_monthly.csv` | US/EUR/JPN macro series — CPI, policy rates, unemployment | Monthly |
| `ons_monthly.csv` | UK CPI headline, core, services, unemployment | Monthly |
| `ecb_hicp_monthly.csv` | Eurozone HICP — headline, core, services, energy | Monthly |
| `ecb_unemployment_monthly.csv` | Eurozone unemployment rate | Monthly |
| `gdp_quarterly.csv` | US real GDP | Quarterly |
| `ons_gdp_quarterly.csv` | UK real GDP | Quarterly |
| `imf_output_gap_annual.csv` | Output gap — US, UK, Germany, Japan | Annual |
| `data_quality_log.csv` | Missing values, coverage, frequency by file | — |


In [1]:
# ============================================================
# Imports and configuration
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import fredapi
import yfinance as yf
from dotenv import load_dotenv
import os
import time
from datetime import datetime

# Load FRED API key from .env
load_dotenv()
FRED_API_KEY = os.environ.get("FRED_API_KEY")
fred = fredapi.Fred(api_key=FRED_API_KEY)

# Date range — fixed cutoff for reproducibility
START_DATE = "2015-01-01"
END_DATE   = "2026-05-01"  # Data as of 1 May 2026

print(f"Configuration loaded")
print(f"FRED API key: {'OK' if FRED_API_KEY else 'MISSING — check .env'}")
print(f"Sample period: {START_DATE} to {END_DATE}")

Configuration loaded
FRED API key: OK
Sample period: 2015-01-01 to 2026-05-01


## 2. FRED Series Definition

All series validated against FRED API prior to data pull. Key data limitations identified during validation:

- **ICE BofA credit spreads (US IG, US HY, EUR HY):** Full history restricted to April 2023 onwards on FRED - the pre-2023 history sits behind the ICE BofA paywall. Credit spread analysis in Notebook 3 is therefore limited to April 2023–May 2026. This does not affect the core sovereign yield, real yield, or Taylor Rule analysis.
- **UK IG OAS (Sterling Corporate):** No series currently available on FRED. Acknowledged as a data gap - UK credit spread analysis is omitted.
- **Eurozone credit:** EUR HY OAS available from April 2023 via `BAMLHE00EHYIOAS`. EUR IG OAS not available - omitted.

In [2]:
# ============================================================
# Validated FRED series dictionary
# All series confirmed active as of May 2026
# ============================================================

FRED_SERIES = {
    # --- US Yields ---
    "US_2Y":                "DGS2",
    "US_5Y":                "DGS5",
    "US_10Y":               "DGS10",
    "US_30Y":               "DGS30",
    # --- US TIPS Real Yields ---
    "US_TIPS_5Y":           "DFII5",
    "US_TIPS_10Y":          "DFII10",
    "US_TIPS_30Y":          "DFII30",
    # --- US Breakevens ---
    "US_BREAKEVEN_10Y":     "T10YIE",
    # --- US Macro ---
    "US_CPI_HEADLINE":      "CPIAUCSL",
    "US_CPI_CORE":          "CPILFESL",
    "US_PCE":               "PCEPI",
    "US_PCE_CORE":          "PCEPILFE",
    "US_UNEMPLOYMENT":      "UNRATE",
    "US_GDP_REAL":          "GDPC1",
    "US_FED_FUNDS":         "FEDFUNDS",
    "US_FED_FUNDS_DAILY":   "DFF",
    # --- Eurozone ---
    "EUR_BUND_10Y":         "IRLTLT01DEM156N",
    "EUR_ECB_RATE":         "ECBDFR",
    "EUR_CPI":              "CP0000EZ19M086NEST",
    "EUR_UNEMPLOYMENT":     "LRHUTTTTEZM156S",
    # --- Japan ---
    "JPN_JGB_10Y":          "IRLTLT01JPM156N",
    "JPN_BOJ_RATE":         "IRSTCB01JPM156N",
    "JPN_CPI":              "JPNCPIALLMINMEI",
    # --- FX ---
    "FX_GBPUSD":            "DEXUSUK",
    "FX_EURUSD":            "DEXUSEU",
    "FX_USDJPY":            "DEXJPUS",
    # --- Market / Risk ---
    "VIX":                  "VIXCLS",
    "BRENT":                "DCOILBRENTEU",
    # --- Credit Spreads (Apr 2023 onwards only — see methodology note above) ---
    "US_IG_OAS":            "BAMLC0A0CM",
    "US_HY_OAS":            "BAMLH0A0HYM2",
    "EUR_HY_OAS":           "BAMLHE00EHYIOAS",
    # UK IG OAS — not available on FRED, omitted
}

print(f"FRED series dictionary loaded — {len(FRED_SERIES)} series defined")

FRED series dictionary loaded — 31 series defined


In [3]:
# ============================================================
# FRED Data Pull
# Pull all 31 series, store in dictionary of DataFrames
# Small delay between requests to avoid rate limiting
# ============================================================

raw_fred = {}
failed = []

print(f"Pulling {len(FRED_SERIES)} series from FRED...\n")

for label, series_id in FRED_SERIES.items():
    try:
        s = fred.get_series(series_id, observation_start=START_DATE, observation_end=END_DATE)
        s.name = label
        raw_fred[label] = s
        print(f"  ✓ {label:<25} {len(s)} observations")
        time.sleep(0.3)
    except Exception as e:
        failed.append((label, series_id, str(e)))
        print(f"  ✗ {label:<25} FAILED — {e}")

print(f"\n{'='*50}")
print(f"Pull complete — {len(raw_fred)} series retrieved, {len(failed)} failed")

if failed:
    print("\nFailed series:")
    for label, series_id, error in failed:
        print(f"  {label} ({series_id}): {error}")

Pulling 31 series from FRED...

  ✓ US_2Y                     2956 observations
  ✓ US_5Y                     2956 observations
  ✓ US_10Y                    2956 observations
  ✓ US_30Y                    2956 observations
  ✓ US_TIPS_5Y                2956 observations
  ✓ US_TIPS_10Y               2956 observations
  ✓ US_TIPS_30Y               2956 observations
  ✓ US_BREAKEVEN_10Y          2957 observations
  ✓ US_CPI_HEADLINE           135 observations
  ✓ US_CPI_CORE               135 observations
  ✓ US_PCE                    135 observations
  ✓ US_PCE_CORE               135 observations
  ✓ US_UNEMPLOYMENT           135 observations
  ✓ US_GDP_REAL               45 observations
  ✓ US_FED_FUNDS              136 observations
  ✓ US_FED_FUNDS_DAILY        4138 observations
  ✓ EUR_BUND_10Y              135 observations
  ✓ EUR_ECB_RATE              4139 observations
  ✓ EUR_CPI                   135 observations
  ✓ EUR_UNEMPLOYMENT          97 observations
  ✓ JPN_JGB_10Y     

In [4]:
# ============================================================
# Organise series into themed DataFrames by frequency
# ============================================================

# --- Daily: Yields ---
yields_daily = pd.DataFrame({
    "US_2Y":    raw_fred["US_2Y"],
    "US_5Y":    raw_fred["US_5Y"],
    "US_10Y":   raw_fred["US_10Y"],
    "US_30Y":   raw_fred["US_30Y"],
    "EUR_BUND_10Y": raw_fred["EUR_BUND_10Y"],
    "JPN_JGB_10Y":  raw_fred["JPN_JGB_10Y"],
})

# --- Daily: TIPS Real Yields and Breakevens ---
real_yields_daily = pd.DataFrame({
    "US_TIPS_5Y":       raw_fred["US_TIPS_5Y"],
    "US_TIPS_10Y":      raw_fred["US_TIPS_10Y"],
    "US_TIPS_30Y":      raw_fred["US_TIPS_30Y"],
    "US_BREAKEVEN_10Y": raw_fred["US_BREAKEVEN_10Y"],
})

# --- Daily: FX ---
fx_daily = pd.DataFrame({
    "GBPUSD": raw_fred["FX_GBPUSD"],
    "EURUSD": raw_fred["FX_EURUSD"],
    "USDJPY": raw_fred["FX_USDJPY"],
})

# --- Daily: Credit Spreads (Apr 2023 onwards) ---
credit_daily = pd.DataFrame({
    "US_IG_OAS":  raw_fred["US_IG_OAS"],
    "US_HY_OAS":  raw_fred["US_HY_OAS"],
    "EUR_HY_OAS": raw_fred["EUR_HY_OAS"],
})

# --- Daily: Market / Risk ---
market_daily = pd.DataFrame({
    "VIX":   raw_fred["VIX"],
    "BRENT": raw_fred["BRENT"],
    "US_FED_FUNDS_DAILY": raw_fred["US_FED_FUNDS_DAILY"],
    "EUR_ECB_RATE":       raw_fred["EUR_ECB_RATE"],
})

# --- Monthly: Macro ---
macro_monthly = pd.DataFrame({
    "US_CPI":          raw_fred["US_CPI_HEADLINE"],
    "US_CPI_CORE":     raw_fred["US_CPI_CORE"],
    "US_PCE":          raw_fred["US_PCE"],
    "US_PCE_CORE":     raw_fred["US_PCE_CORE"],
    "US_UNEMPLOYMENT": raw_fred["US_UNEMPLOYMENT"],
    "US_FED_FUNDS":    raw_fred["US_FED_FUNDS"],
    "EUR_CPI":         raw_fred["EUR_CPI"],
    "EUR_UNEMPLOYMENT":raw_fred["EUR_UNEMPLOYMENT"],
    "EUR_BUND_10Y":    raw_fred["EUR_BUND_10Y"],
    "JPN_CPI":         raw_fred["JPN_CPI"],
    "JPN_JGB_10Y":     raw_fred["JPN_JGB_10Y"],
    "JPN_BOJ_RATE":    raw_fred["JPN_BOJ_RATE"],
})

# --- Quarterly: GDP ---
gdp_quarterly = pd.DataFrame({
    "US_GDP_REAL": raw_fred["US_GDP_REAL"],
})

print("DataFrames created:")
print(f"  yields_daily:      {yields_daily.shape}")
print(f"  real_yields_daily: {real_yields_daily.shape}")
print(f"  fx_daily:          {fx_daily.shape}")
print(f"  credit_daily:      {credit_daily.shape}")
print(f"  market_daily:      {market_daily.shape}")
print(f"  macro_monthly:     {macro_monthly.shape}")
print(f"  gdp_quarterly:     {gdp_quarterly.shape}")

DataFrames created:
  yields_daily:      (2995, 6)
  real_yields_daily: (2957, 4)
  fx_daily:          (2952, 3)
  credit_daily:      (794, 3)
  market_daily:      (4139, 4)
  macro_monthly:     (136, 12)
  gdp_quarterly:     (45, 1)


In [5]:
# ============================================================
# Save processed CSVs to /data/processed/
# ============================================================

import os

OUTPUT_DIR = "../data/processed/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

dataframes = {
    "yields_daily.csv":      yields_daily,
    "real_yields_daily.csv": real_yields_daily,
    "fx_daily.csv":          fx_daily,
    "credit_daily.csv":      credit_daily,
    "market_daily.csv":      market_daily,
    "macro_monthly.csv":     macro_monthly,
    "gdp_quarterly.csv":     gdp_quarterly,
}

print("Saving processed CSVs...\n")
for filename, df in dataframes.items():
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path)
    print(f"  ✓ {filename:<35} {df.shape[0]} rows x {df.shape[1]} cols")

print(f"\nAll files saved to {OUTPUT_DIR}")

Saving processed CSVs...

  ✓ yields_daily.csv                    2995 rows x 6 cols
  ✓ real_yields_daily.csv               2957 rows x 4 cols
  ✓ fx_daily.csv                        2952 rows x 3 cols
  ✓ credit_daily.csv                    794 rows x 3 cols
  ✓ market_daily.csv                    4139 rows x 4 cols
  ✓ macro_monthly.csv                   136 rows x 12 cols
  ✓ gdp_quarterly.csv                   45 rows x 1 cols

All files saved to ../data/processed/


## Bank of England Data Pull

UK gilt spot curves (nominal, real, inflation breakeven) read from BoE Excel files in `/data/raw/`.
Three files per curve type concatenated into continuous series from January 2015 to May 2026.
Only 2Y, 5Y, 10Y and 30Y maturities extracted.

In [14]:
# ============================================================
# Bank of England Gilt Curve Data
# Read and concatenate Excel files for nominal, real, inflation
# Maturities differ by curve type - real and inflation curves
# start at 2.5Y (no short-maturity index-linked gilts issued)
# 30Y unavailable pre-2016 — NaN for those dates
# ============================================================

BOE_RAW_DIR = "../data/raw/"
SPOT_SHEET  = "4. spot curve"

# Target maturities per curve type
BOE_MATURITIES = {
    "nominal":   {"standard": [2.0, 5.0, 10.0, 30.0], "early": [2.0, 5.0, 10.0]},
    "real":      {"standard": [5.0, 10.0, 30.0],       "early": [5.0, 10.0]},
    "inflation": {"standard": [5.0, 10.0, 30.0],       "early": [5.0, 10.0]},
}

def read_boe_file(filepath, target_maturities):
    """
    Read a single BoE yield curve Excel file from the spot curve tab.

    Structure:
        Row 3: Maturity values in years
        Row 5+: Date index with yield values across columns

    Returns DataFrame with datetime index and target maturity columns.
    """
    df = pd.read_excel(filepath, sheet_name=SPOT_SHEET, header=None)
    maturity_row = df.iloc[3, 1:].astype(float)

    target_cols = []
    for mat in target_maturities:
        matches = maturity_row[maturity_row == mat].index.tolist()
        if matches:
            target_cols.append(matches[0])
        else:
            raise ValueError(f"Maturity {mat}Y not found in {filepath}")

    data = df.iloc[5:, [0] + target_cols].copy()
    data.columns = ["date"] + [f"{int(m)}Y" if m == int(m) else f"{m}Y"
                                for m in target_maturities]
    data["date"] = pd.to_datetime(data["date"], dayfirst=True, errors="coerce")
    data = data.dropna(subset=["date"])
    data = data.set_index("date")

    for col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

    return data


def load_boe_curve(curve_type, file_suffixes):
    """
    Load and concatenate BoE Excel files for a given curve type.
    Handles pre-2016 files where max maturity is 25Y — 30Y column
    will be NaN for those dates.

    Parameters:
        curve_type:    str — 'nominal', 'real', or 'inflation'
        file_suffixes: list of str — in chronological order

    Returns concatenated DataFrame filtered to START_DATE to END_DATE.
    """
    mats        = BOE_MATURITIES[curve_type]
    standard    = mats["standard"]
    early       = mats["early"]
    all_cols    = [f"{int(m)}Y" if m == int(m) else f"{m}Y" for m in standard]
    dfs         = []

    for suffix in file_suffixes:
        filepath = os.path.join(BOE_RAW_DIR, f"boe_{curve_type}_{suffix}.xlsx")

        # Check max maturity available
        probe   = pd.read_excel(filepath, sheet_name=SPOT_SHEET, header=None)
        max_mat = probe.iloc[3, 1:].astype(float).max()

        target_mats = standard if max_mat >= 30.0 else early
        df = read_boe_file(filepath, target_mats)

        # Add 30Y as NaN if not available
        for col in all_cols:
            if col not in df.columns:
                df[col] = float("nan")

        df = df[all_cols]
        dfs.append(df)

        print(f"  ✓ boe_{curve_type}_{suffix}.xlsx — {len(df)} rows "
              f"({df.index.min().date()} to {df.index.max().date()}) "
              f"| 30Y: {'available' if max_mat >= 30 else 'NaN — pre-2016'}")

    combined = pd.concat(dfs)
    combined = combined[~combined.index.duplicated(keep="last")]
    combined = combined.sort_index()
    combined = combined[combined.index >= START_DATE]
    combined = combined[combined.index <= END_DATE]

    return combined


FILE_SUFFIXES = ["2005_2015", "2016_2024", "2025_present", "current"]

print("Loading BoE nominal gilt curve...")
boe_nominal = load_boe_curve("nominal", FILE_SUFFIXES)
print(f"  → Combined: {len(boe_nominal)} rows "
      f"({boe_nominal.index.min().date()} to {boe_nominal.index.max().date()})\n")

print("Loading BoE real gilt curve...")
boe_real = load_boe_curve("real", FILE_SUFFIXES)
print(f"  → Combined: {len(boe_real)} rows "
      f"({boe_real.index.min().date()} to {boe_real.index.max().date()})\n")

print("Loading BoE inflation breakeven curve...")
boe_inflation = load_boe_curve("inflation", FILE_SUFFIXES)
print(f"  → Combined: {len(boe_inflation)} rows "
      f"({boe_inflation.index.min().date()} to {boe_inflation.index.max().date()})\n")

print("=" * 50)
print("BoE data load complete")
print(f"  Nominal:   {boe_nominal.shape}")
print(f"  Real:      {boe_real.shape}")
print(f"  Inflation: {boe_inflation.shape}")


Loading BoE nominal gilt curve...
  ✓ boe_nominal_2005_2015.xlsx — 2869 rows (2005-01-03 to 2015-12-31) | 30Y: NaN — pre-2016
  ✓ boe_nominal_2016_2024.xlsx — 2348 rows (2016-01-01 to 2024-12-31) | 30Y: available
  ✓ boe_nominal_2025_present.xlsx — 325 rows (2025-01-01 to 2026-03-31) | 30Y: available
  ✓ boe_nominal_current.xlsx — 22 rows (2026-04-01 to 2026-04-30) | 30Y: available
  → Combined: 2956 rows (2015-01-01 to 2026-04-30)

Loading BoE real gilt curve...
  ✓ boe_real_2005_2015.xlsx — 2869 rows (2005-01-03 to 2015-12-31) | 30Y: NaN — pre-2016
  ✓ boe_real_2016_2024.xlsx — 2348 rows (2016-01-01 to 2024-12-31) | 30Y: available
  ✓ boe_real_2025_present.xlsx — 325 rows (2025-01-01 to 2026-03-31) | 30Y: available
  ✓ boe_real_current.xlsx — 22 rows (2026-04-01 to 2026-04-30) | 30Y: available
  → Combined: 2956 rows (2015-01-01 to 2026-04-30)

Loading BoE inflation breakeven curve...
  ✓ boe_inflation_2005_2015.xlsx — 2869 rows (2005-01-03 to 2015-12-31) | 30Y: NaN — pre-2016
  ✓ bo

In [15]:
# ============================================================
# Save BoE processed CSVs
# ============================================================

boe_files = {
    "boe_nominal_daily.csv":   boe_nominal,
    "boe_real_daily.csv":      boe_real,
    "boe_inflation_daily.csv": boe_inflation,
}

print("Saving BoE processed CSVs...\n")
for filename, df in boe_files.items():
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path)
    print(f"  ✓ {filename:<35} {df.shape[0]} rows x {df.shape[1]} cols")

print(f"\nBoE files saved to {OUTPUT_DIR}")

Saving BoE processed CSVs...

  ✓ boe_nominal_daily.csv               2956 rows x 4 cols
  ✓ boe_real_daily.csv                  2956 rows x 3 cols
  ✓ boe_inflation_daily.csv             2956 rows x 3 cols

BoE files saved to ../data/processed/


## ONS Data Pull

UK macroeconomic data read from ONS CSV downloads saved to `/data/raw/`.
All series share a common structure — 8 header rows followed by yearly, quarterly and monthly data.
Monthly frequency extracted for CPI and unemployment series. Quarterly for GDP.

**Series:**
- D7BT — CPI All Items (2015=100)
- DKC6 — CPI Core excl. energy, food, alcohol, tobacco (2015=100)
- D7F5 — CPI Services
- MGSX — Unemployment rate, 16+, seasonally adjusted
- ABMI — GDP chained volume measure, seasonally adjusted

In [18]:
# ============================================================
# ONS Data Pull
# Read from manually downloaded CSVs in /data/raw/
# 8 header rows skipped — monthly and quarterly data extracted
# Monthly date format: "2021 JUN" | Quarterly: "2025 Q1"
# ============================================================

ONS_RAW_DIR = "../data/raw/"

ONS_FILES = {
    "UK_CPI":          ("ons_cpi.csv",          "monthly"),
    "UK_CPI_CORE":     ("ons_cpi_core.csv",      "monthly"),
    "UK_CPI_SERVICES": ("ons_cpi_services.csv",  "monthly"),
    "UK_UNEMPLOYMENT": ("ons_unemployment.csv",  "monthly"),
    "UK_GDP":          ("ons_gdp.csv",           "quarterly"),
}

def parse_ons_date(date_str, freq):
    """
    Parse ONS date strings to pandas Timestamp.
    Monthly format: '2021 JUN' -> 2021-06-01
    Quarterly format: '2025 Q1' -> 2025-01-01
    """
    date_str = str(date_str).strip()
    if freq == "monthly":
        return pd.to_datetime(date_str, format="%Y %b")
    elif freq == "quarterly":
        year, q = date_str.split(" Q")
        month = (int(q) - 1) * 3 + 1
        return pd.Timestamp(year=int(year), month=month, day=1)
    else:
        raise ValueError(f"Unknown frequency: {freq}")

def read_ons_file(filename, freq):
    """
    Read a single ONS timeseries CSV.
    Skips 8 header rows, extracts date and value columns.
    Filters to monthly or quarterly rows based on date format.
    Returns pandas Series with datetime index filtered to project date range.
    """
    filepath = os.path.join(ONS_RAW_DIR, filename)
    df = pd.read_csv(filepath, skiprows=8, header=0)

    # ONS CSVs have two columns — date and value
    df.columns = ["date", "value"]
    df = df.dropna(subset=["date", "value"])

    # Filter to correct frequency rows based on date format
    if freq == "monthly":
        # Monthly rows contain month abbreviation e.g. "2021 JUN"
        mask = df["date"].astype(str).str.match(r"^\d{4}\s+[A-Z]{3}$")
    else:
        # Quarterly rows contain "Q" e.g. "2025 Q1"
        mask = df["date"].astype(str).str.contains("Q")

    df = df[mask].copy()

    # Parse dates and values
    df["date"]  = df["date"].apply(lambda x: parse_ons_date(x, freq))
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["date", "value"])
    df = df.set_index("date")["value"]
    df = df.sort_index()
    df = df[df.index >= START_DATE]
    df = df[df.index <= END_DATE]

    return df

raw_ons = {}
failed_ons = []

print(f"Reading {len(ONS_FILES)} ONS series...\n")

for label, (filename, freq) in ONS_FILES.items():
    try:
        s = read_ons_file(filename, freq)
        s.name = label
        raw_ons[label] = s
        print(f"  ✓ {label:<25} {len(s)} observations ({freq}) "
              f"| {s.index.min().date()} to {s.index.max().date()}")
    except Exception as e:
        failed_ons.append((label, filename, str(e)))
        print(f"  ✗ {label:<25} FAILED — {e}")

print(f"\n{'='*50}")
print(f"ONS read complete — {len(raw_ons)} series loaded, {len(failed_ons)} failed")

Reading 5 ONS series...

  ✓ UK_CPI                    135 observations (monthly) | 2015-01-01 to 2026-03-01
  ✓ UK_CPI_CORE               135 observations (monthly) | 2015-01-01 to 2026-03-01
  ✓ UK_CPI_SERVICES           135 observations (monthly) | 2015-01-01 to 2026-03-01
  ✓ UK_UNEMPLOYMENT           133 observations (monthly) | 2015-01-01 to 2026-01-01
  ✓ UK_GDP                    44 observations (quarterly) | 2015-01-01 to 2025-10-01

ONS read complete — 5 series loaded, 0 failed


In [19]:
# ============================================================
# Save ONS processed CSVs
# ============================================================

ons_monthly = pd.DataFrame({
    "UK_CPI":          raw_ons["UK_CPI"],
    "UK_CPI_CORE":     raw_ons["UK_CPI_CORE"],
    "UK_CPI_SERVICES": raw_ons["UK_CPI_SERVICES"],
    "UK_UNEMPLOYMENT": raw_ons["UK_UNEMPLOYMENT"],
})

ons_gdp = pd.DataFrame({
    "UK_GDP": raw_ons["UK_GDP"],
})

ons_files = {
    "ons_monthly.csv":     ons_monthly,
    "ons_gdp_quarterly.csv": ons_gdp,
}

print("Saving ONS processed CSVs...\n")
for filename, df in ons_files.items():
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path)
    print(f"  ✓ {filename:<35} {df.shape[0]} rows x {df.shape[1]} cols")

print(f"\nONS files saved to {OUTPUT_DIR}")

Saving ONS processed CSVs...

  ✓ ons_monthly.csv                     135 rows x 4 cols
  ✓ ons_gdp_quarterly.csv               44 rows x 1 cols

ONS files saved to ../data/processed/


In [20]:
# ============================================================
# yfinance Pull — Equity Indices and Commodities
# Daily adjusted close prices
# ============================================================

YFINANCE_TICKERS = {
    "SP500":      "^GSPC",
    "FTSE100":    "^FTSE",
    "EUROSTOXX":  "^STOXX50E",
    "NIKKEI":     "^N225",
    "GOLD":       "GC=F",
    "BRENT_YF":   "BZ=F",
}

print(f"Pulling {len(YFINANCE_TICKERS)} tickers from yfinance...\n")

raw_yf = {}
failed_yf = []

for label, ticker in YFINANCE_TICKERS.items():
    try:
        df = yf.download(
            ticker,
            start=START_DATE,
            end=END_DATE,
            progress=False,
            auto_adjust=True
        )
        if df.empty:
            raise ValueError("Empty DataFrame returned")
        s = df["Close"].squeeze()
        s.name = label
        s.index = pd.to_datetime(s.index)
        raw_yf[label] = s
        print(f"  ✓ {label:<15} {len(s)} observations "
              f"| {s.index.min().date()} to {s.index.max().date()}")
    except Exception as e:
        failed_yf.append((label, ticker, str(e)))
        print(f"  ✗ {label:<15} FAILED — {e}")

print(f"\n{'='*50}")
print(f"yfinance pull complete — {len(raw_yf)} series retrieved, {len(failed_yf)} failed")

Pulling 6 tickers from yfinance...

  ✓ SP500           2848 observations | 2015-01-02 to 2026-04-30
  ✓ FTSE100         2861 observations | 2015-01-02 to 2026-04-30
  ✓ EUROSTOXX       2845 observations | 2015-01-05 to 2026-04-30
  ✓ NIKKEI          2767 observations | 2015-01-05 to 2026-04-30
  ✓ GOLD            2847 observations | 2015-01-02 to 2026-04-30
  ✓ BRENT_YF        2849 observations | 2015-01-02 to 2026-04-30

yfinance pull complete — 6 series retrieved, 0 failed


In [21]:
# ============================================================
# Save yfinance processed CSV
# ============================================================

equities_commodities = pd.DataFrame({
    "SP500":     raw_yf["SP500"],
    "FTSE100":   raw_yf["FTSE100"],
    "EUROSTOXX": raw_yf["EUROSTOXX"],
    "NIKKEI":    raw_yf["NIKKEI"],
    "GOLD":      raw_yf["GOLD"],
    "BRENT_YF":  raw_yf["BRENT_YF"],
})

path = os.path.join(OUTPUT_DIR, "equities_commodities_daily.csv")
equities_commodities.to_csv(path)

print(f"  ✓ equities_commodities_daily.csv — "
      f"{equities_commodities.shape[0]} rows x {equities_commodities.shape[1]} cols")
print(f"\nSaved to {OUTPUT_DIR}")

  ✓ equities_commodities_daily.csv — 2945 rows x 6 cols

Saved to ../data/processed/


## IMF Output Gap Data

Output gap estimates (% of potential GDP) downloaded from IMF World Economic Outlook database.
April 2026 vintage. Annual frequency - covers 2015 to 2026 including IMF forward projections.
Data for US, UK, Germany (Eurozone proxy) and Japan.

**Methodology note:** Output gap estimates are uncertain and subject to revision between WEO vintages.
Results of Taylor Rule analysis in Notebook 2 are sensitive to the vintage used - April 2026 
vintage recorded here for reproducibility.

In [23]:
# ============================================================
# IMF Output Gap Data
# WEO April 2026 vintage — annual frequency
# Sideways format — countries in rows, years in columns
# Columns A-G are metadata, data begins column H (1980)
# ============================================================

IMF_FILE = "../data/raw/IMF_output_gap.csv"

# Row indices (0-based, after header row 0)
IMF_COUNTRIES = {
    "UK":      0,  # row 2 in Excel = index 0 after header
    "US":      1,  # row 3
    "Germany": 2,  # row 4
    "Japan":   3,  # row 5
}

def read_imf_output_gap(filepath, countries, start_year=2015):
    """
    Read IMF WEO output gap CSV.
    Format: sideways table — countries in rows, years in columns.
    First 7 columns are metadata — data starts column index 7.
    
    Returns DataFrame with year as index, countries as columns.
    """
    df = pd.read_csv(filepath, header=0, low_memory=False)
    
    # Extract year columns — everything from column 7 onwards
    year_cols = df.columns[7:]
    
    result = {}
    for country, row_idx in countries.items():
        row = df.iloc[row_idx, 7:]
        row.index = year_cols.astype(int)
        row = pd.to_numeric(row, errors="coerce")
        result[country] = row

    output = pd.DataFrame(result)
    output.index.name = "year"
    output = output[output.index >= start_year]
    return output

imf_output_gap = read_imf_output_gap(IMF_FILE, IMF_COUNTRIES)

print("IMF Output Gap (% of potential GDP)\n")
print(imf_output_gap.to_string())
print(f"\nShape: {imf_output_gap.shape}")
print(f"Years: {imf_output_gap.index.min()} to {imf_output_gap.index.max()}")

IMF Output Gap (% of potential GDP)

         UK     US  Germany  Japan
year                              
2015 -1.167 -1.062   -0.324 -0.186
2016 -0.746 -1.468    0.095  0.095
2017  0.904 -1.287    1.040  1.065
2018  1.598 -0.598    0.883  1.911
2019  1.528  0.054    0.438  0.672
2020 -3.475 -3.380   -3.085 -2.950
2021  0.458  0.127   -0.762 -1.302
2022  2.246  0.136    1.315 -0.625
2023 -0.270  0.290   -0.118 -0.009
2024 -0.532  0.385   -1.049 -0.042
2025 -0.565  0.001   -1.148  0.420
2026 -0.922 -0.001   -0.882  0.400
2027 -0.880  0.000   -0.426  0.179
2028 -0.619  0.000    0.045  0.017
2029 -0.363  0.000    0.257    NaN
2030 -0.180 -0.001    0.168    NaN
2031 -0.126  0.000    0.047    NaN

Shape: (17, 4)
Years: 2015 to 2031


In [24]:
# ============================================================
# Save IMF output gap — filter to project sample 2015-2026
# ============================================================

imf_output_gap_save = imf_output_gap[imf_output_gap.index <= 2026]

path = os.path.join(OUTPUT_DIR, "imf_output_gap_annual.csv")
imf_output_gap_save.to_csv(path)

print(f"  ✓ imf_output_gap_annual.csv — "
      f"{imf_output_gap_save.shape[0]} rows x {imf_output_gap_save.shape[1]} cols")
print(f"\nSaved to {OUTPUT_DIR}")

  ✓ imf_output_gap_annual.csv — 12 rows x 4 cols

Saved to ../data/processed/


## ECB HICP Data

Eurozone HICP inflation components downloaded from ECB Statistical Data Warehouse.
Four series: headline, core, services and energy - all annual rate of change, monthly frequency.
Covers Eurozone aggregate (not Germany specifically - consistent with using Eurozone macro
data alongside German Bund as sovereign benchmark).

In [30]:
# ============================================================
# ECB HICP Data
# Downloaded from ECB Statistical Data Warehouse (sdw.ecb.europa.eu)
# Format: 2 header rows, date in col B (format: 2011Mar), value in col C
# Annual rate of change, monthly frequency
# ============================================================

ECB_FILES = {
    "EUR_HICP_HEADLINE": "ecb_hicp_headline.csv",
    "EUR_HICP_CORE":     "ecb_hicp_core.csv",
    "EUR_HICP_SERVICES": "ecb_hicp_services.csv",
    "EUR_HICP_ENERGY":   "ecb_hicp_energy.csv",
}

def read_ecb_file(filename):
    """
    Read a single ECB SDW HICP CSV.
    Structure: 2 header rows, date in column B, value in column C.
    Date format: '2011Mar' -> 2011-03-01.
    Returns pandas Series with datetime index filtered to project range.
    """
    filepath = os.path.join(ECB_FILES_DIR, filename)
    df = pd.read_csv(filepath, skiprows=2, header=None)
    
    # Extract date and value columns (B=1, C=2)
    df = df.iloc[:, [1, 2]].copy()
    df.columns = ["date", "value"]
    df = df.dropna(subset=["date", "value"])
    
    # Parse date format '2011Mar'
    df["date"]  = pd.to_datetime(df["date"].astype(str).str.strip(), format="%Y%b")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["date", "value"])
    df = df.set_index("date")["value"]
    df = df.sort_index()
    df = df[df.index >= START_DATE]
    df = df[df.index <= END_DATE]
    
    return df

ECB_FILES_DIR = "../data/raw/"
raw_ecb = {}
failed_ecb = []

print(f"Reading {len(ECB_FILES)} ECB HICP series...\n")

for label, filename in ECB_FILES.items():
    try:
        s = read_ecb_file(filename)
        s.name = label
        raw_ecb[label] = s
        print(f"  ✓ {label:<25} {len(s)} observations "
              f"| {s.index.min().date()} to {s.index.max().date()}")
    except Exception as e:
        failed_ecb.append((label, filename, str(e)))
        print(f"  ✗ {label:<25} FAILED — {e}")

print(f"\n{'='*50}")
print(f"ECB read complete — {len(raw_ecb)} series loaded, {len(failed_ecb)} failed")

Reading 4 ECB HICP series...

  ✓ EUR_HICP_HEADLINE         136 observations | 2015-01-01 to 2026-04-01
  ✓ EUR_HICP_CORE             136 observations | 2015-01-01 to 2026-04-01
  ✓ EUR_HICP_SERVICES         136 observations | 2015-01-01 to 2026-04-01
  ✓ EUR_HICP_ENERGY           136 observations | 2015-01-01 to 2026-04-01

ECB read complete — 4 series loaded, 0 failed


In [31]:
# ============================================================
# Save ECB HICP processed CSV
# ============================================================

ecb_hicp = pd.DataFrame({
    "EUR_HICP_HEADLINE": raw_ecb["EUR_HICP_HEADLINE"],
    "EUR_HICP_CORE":     raw_ecb["EUR_HICP_CORE"],
    "EUR_HICP_SERVICES": raw_ecb["EUR_HICP_SERVICES"],
    "EUR_HICP_ENERGY":   raw_ecb["EUR_HICP_ENERGY"],
})

path = os.path.join(OUTPUT_DIR, "ecb_hicp_monthly.csv")
ecb_hicp.to_csv(path)

print(f"  ✓ ecb_hicp_monthly.csv — "
      f"{ecb_hicp.shape[0]} rows x {ecb_hicp.shape[1]} cols")
print(f"\nSaved to {OUTPUT_DIR}")

  ✓ ecb_hicp_monthly.csv — 136 rows x 4 cols

Saved to ../data/processed/


## ECB Supplementary Data

Additional ECB series from the ECB Statistical Data Warehouse:
- Eurozone unemployment rate (age 15-74, seasonally adjusted) - to fill FRED gap post-2023
- ECB AAA yield curve (2Y, 5Y, 10Y, 30Y) - for yield curve shape and rate differential analysis
  Note: AAA curve is NOT a pure German Bund series - see methodology note in project diary

In [35]:
# ============================================================
# ECB Supplementary Data
# Three file formats handled separately:
# HICP: 2 header rows, date format '2011Mar'
# Unemployment: 1 header row, date format '2000Jan'
# Yield curve: 1 header row, date format '06 Sep 2004'
# ============================================================

# --- Unemployment ---
def read_ecb_unemployment(filename):
    """
    Read ECB unemployment CSV.
    1 header row, date in col B format '2000Jan', value in col C.
    Returns pandas Series filtered to project date range.
    """
    filepath = os.path.join(ECB_FILES_DIR, filename)
    df = pd.read_csv(filepath, skiprows=1, header=None)
    df = df.iloc[:, [1, 2]].copy()
    df.columns = ["date", "value"]
    df = df.dropna(subset=["date", "value"])
    df["date"]  = pd.to_datetime(df["date"].astype(str).str.strip(), format="%Y%b")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["date", "value"])
    df = df.set_index("date")["value"]
    df = df.sort_index()
    df = df[df.index >= START_DATE]
    df = df[df.index <= END_DATE]
    return df

# --- Yield curve ---
def read_ecb_yield(filename):
    """
    Read ECB AAA yield curve CSV.
    1 header row, date in col B format '06 Sep 2004', value in col C.
    Returns pandas Series filtered to project date range.
    """
    filepath = os.path.join(ECB_FILES_DIR, filename)
    df = pd.read_csv(filepath, skiprows=1, header=None)
    df = df.iloc[:, [1, 2]].copy()
    df.columns = ["date", "value"]
    df = df.dropna(subset=["date", "value"])
    df["date"]  = pd.to_datetime(df["date"].astype(str).str.strip(), format="%d %b %Y")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["date", "value"])
    df = df.set_index("date")["value"]
    df = df.sort_index()
    df = df[df.index >= START_DATE]
    df = df[df.index <= END_DATE]
    return df

# --- Pull unemployment ---
print("Reading ECB unemployment...")
ecb_unemployment = read_ecb_unemployment("ecb_unemployment.csv")
ecb_unemployment.name = "EUR_UNEMPLOYMENT_ECB"
print(f"  ✓ EUR_UNEMPLOYMENT_ECB     {len(ecb_unemployment)} observations "
      f"| {ecb_unemployment.index.min().date()} to {ecb_unemployment.index.max().date()}")

# --- Pull yield curve ---
ECB_YIELD_FILES = {
    "EUR_AAA_2Y":  "ecb_yield_2y.csv",
    "EUR_AAA_5Y":  "ecb_yield_5y.csv",
    "EUR_AAA_10Y": "ecb_yield_10y.csv",
    "EUR_AAA_30Y": "ecb_yield_30y.csv",
}

print("\nReading ECB AAA yield curve...")
raw_ecb_yields = {}
for label, filename in ECB_YIELD_FILES.items():
    try:
        s = read_ecb_yield(filename)
        s.name = label
        raw_ecb_yields[label] = s
        print(f"  ✓ {label:<20} {len(s)} observations "
              f"| {s.index.min().date()} to {s.index.max().date()}")
    except Exception as e:
        print(f"  ✗ {label:<20} FAILED — {e}")

print(f"\n{'='*50}")
print("ECB supplementary data load complete")

Reading ECB unemployment...
  ✓ EUR_UNEMPLOYMENT_ECB     135 observations | 2015-01-01 to 2026-03-01

Reading ECB AAA yield curve...
  ✓ EUR_AAA_2Y           2889 observations | 2015-01-02 to 2026-04-30
  ✓ EUR_AAA_5Y           2889 observations | 2015-01-02 to 2026-04-30
  ✓ EUR_AAA_10Y          2889 observations | 2015-01-02 to 2026-04-30
  ✓ EUR_AAA_30Y          2889 observations | 2015-01-02 to 2026-04-30

ECB supplementary data load complete


In [36]:
# ============================================================
# Save ECB supplementary processed CSVs
# ============================================================

# ECB AAA yield curve — daily
ecb_yields = pd.DataFrame({
    "EUR_AAA_2Y":  raw_ecb_yields["EUR_AAA_2Y"],
    "EUR_AAA_5Y":  raw_ecb_yields["EUR_AAA_5Y"],
    "EUR_AAA_10Y": raw_ecb_yields["EUR_AAA_10Y"],
    "EUR_AAA_30Y": raw_ecb_yields["EUR_AAA_30Y"],
})

# ECB unemployment — monthly
ecb_unemp = pd.DataFrame({
    "EUR_UNEMPLOYMENT_ECB": ecb_unemployment,
})

ecb_supp_files = {
    "ecb_aaa_yields_daily.csv":   ecb_yields,
    "ecb_unemployment_monthly.csv": ecb_unemp,
}

print("Saving ECB supplementary processed CSVs...\n")
for filename, df in ecb_supp_files.items():
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path)
    print(f"  ✓ {filename:<40} {df.shape[0]} rows x {df.shape[1]} cols")

print(f"\nSaved to {OUTPUT_DIR}")

Saving ECB supplementary processed CSVs...

  ✓ ecb_aaa_yields_daily.csv                 2889 rows x 4 cols
  ✓ ecb_unemployment_monthly.csv             135 rows x 1 cols

Saved to ../data/processed/


In [37]:
# ============================================================
# Data Quality Log
# Loads each processed CSV and documents:
# — shape, date range, missing values per column
# ============================================================

import glob

PROCESSED_DIR = "../data/processed/"

quality_log = []

files_to_check = {
    "yields_daily.csv":              "daily",
    "real_yields_daily.csv":         "daily",
    "fx_daily.csv":                  "daily",
    "credit_daily.csv":              "daily",
    "market_daily.csv":              "daily",
    "boe_nominal_daily.csv":         "daily",
    "boe_real_daily.csv":            "daily",
    "boe_inflation_daily.csv":       "daily",
    "equities_commodities_daily.csv":"daily",
    "macro_monthly.csv":             "monthly",
    "gdp_quarterly.csv":             "quarterly",
    "ons_monthly.csv":               "monthly",
    "ons_gdp_quarterly.csv":         "quarterly",
    "imf_output_gap_annual.csv":     "annual",
    "ecb_hicp_monthly.csv":          "monthly",
    "ecb_aaa_yields_daily.csv":       "daily",
    "ecb_unemployment_monthly.csv":   "monthly",
}

print(f"{'File':<40} {'Freq':<12} {'Rows':<8} {'Cols':<6} "
      f"{'Start':<12} {'End':<12} {'Missing'}")
print("-" * 105)

for filename, freq in files_to_check.items():
    path = os.path.join(PROCESSED_DIR, filename)
    df = pd.read_csv(path, index_col=0, parse_dates=True)

    start     = df.index.min()
    end       = df.index.max()
    total_missing = df.isnull().sum().sum()
    missing_detail = ", ".join([
        f"{col}: {df[col].isnull().sum()}"
        for col in df.columns
        if df[col].isnull().sum() > 0
    ])
    missing_str = missing_detail if missing_detail else "None"

    print(f"{filename:<40} {freq:<12} {len(df):<8} {len(df.columns):<6} "
          f"{str(start.date()):<12} {str(end.date()):<12} {missing_str}")

    quality_log.append({
        "file":            filename,
        "frequency":       freq,
        "rows":            len(df),
        "cols":            len(df.columns),
        "start":           str(start.date()),
        "end":             str(end.date()),
        "total_missing":   total_missing,
        "missing_detail":  missing_str,
    })

# Save log to processed folder
log_df = pd.DataFrame(quality_log)
log_df.to_csv(os.path.join(PROCESSED_DIR, "data_quality_log.csv"), index=False)

print(f"\n{'='*105}")
print(f"Data quality log saved to {PROCESSED_DIR}data_quality_log.csv")

File                                     Freq         Rows     Cols   Start        End          Missing
---------------------------------------------------------------------------------------------------------
yields_daily.csv                         daily        2995     6      2015-01-01   2026-04-30   US_2Y: 162, US_5Y: 162, US_10Y: 162, US_30Y: 162, EUR_BUND_10Y: 2860, JPN_JGB_10Y: 2860
real_yields_daily.csv                    daily        2957     4      2015-01-01   2026-05-01   US_TIPS_5Y: 124, US_TIPS_10Y: 124, US_TIPS_30Y: 124, US_BREAKEVEN_10Y: 123
fx_daily.csv                             daily        2952     3      2015-01-01   2026-04-24   GBPUSD: 125, EURUSD: 125, USDJPY: 125
credit_daily.csv                         daily        794      3      2023-05-02   2026-04-30   US_IG_OAS: 9, US_HY_OAS: 8, EUR_HY_OAS: 8
market_daily.csv                         daily        4139     4      2015-01-01   2026-05-01   VIX: 1262, BRENT: 1266, US_FED_FUNDS_DAILY: 1
boe_nominal_daily.csv

## Data Quality Summary - Key Findings

**Normal missing data (weekends/holidays):** Daily financial series contain expected NaN values 
for non-trading days. These are forward-filled in subsequent notebooks where required for analysis.

**BoE 30Y gilt curve:** 347 missing values reflect the deliberate NaN assignment for pre-2016 dates 
when the BoE curve extended only to 25Y. Analysis using 30Y gilt data is therefore constrained 
to January 2016 onwards.

**ICE BofA credit spreads:** All three credit series begin May 2023 - full history sits behind 
the ICE BofA paywall. Credit spread analysis is limited to May 2023–May 2026.

**Eurozone and Japan monthly series in yields_daily.csv:** EUR_BUND_10Y and JPN_JGB_10Y are 
monthly FRED series - the ~2,860 NaN values reflect daily gaps between monthly observations. 
These series are used from macro_monthly.csv in subsequent notebooks, not yields_daily.csv.

**Japan data quality:** JPN_CPI has 58 missing monthly observations out of 136 - a significant 
gap reflecting the patchiness of Japanese macro data on FRED. Japan inflation decomposition 
in Notebook 2 is therefore less granular than the US, UK and Eurozone analysis. Acknowledged 
as a limitation.

**Eurozone unemployment:** 39 missing monthly observations - some reporting gaps in the FRED 
series. Interpolated where necessary in subsequent notebooks.

## Data Loader

Reusable loader functions written to `src/data_loader.py`.
All subsequent notebooks import from this module rather than handling file paths directly.

In [38]:
# ============================================================
# Write src/data_loader.py
# ============================================================

loader_code = '''"""
data_loader.py
==============
Reusable data loading functions for the Macro Divergence and Sovereign Bond Value project.
All processed CSVs are loaded from /data/processed/ relative to the project root.

Usage in notebooks:
    import sys
    sys.path.append("..")
    from src.data_loader import load_yields, load_macro, load_boe_nominal ...

Author: Thomas Sumner
Date: May 2026
"""

import pandas as pd
import os

# Path to processed data directory
PROCESSED_DIR = os.path.join(os.path.dirname(__file__), "..", "data", "processed")


def _load(filename, **kwargs):
    """Internal helper -- loads a CSV from the processed directory."""
    path = os.path.join(PROCESSED_DIR, filename)
    return pd.read_csv(path, index_col=0, parse_dates=True, **kwargs)


# ---------------------------------------------------------------
# FRED Series
# ---------------------------------------------------------------

def load_yields():
    """
    Nominal sovereign yield curves from FRED.
    Daily frequency. Columns: US_2Y, US_5Y, US_10Y, US_30Y,
    EUR_BUND_10Y, JPN_JGB_10Y.
    Note: EUR and JPN columns are monthly series -- significant NaN gaps.
    Use load_macro() for monthly EUR/JPN yield data.
    """
    return _load("yields_daily.csv")


def load_real_yields():
    """
    US TIPS real yields and 10Y breakeven inflation from FRED.
    Daily frequency. Columns: US_TIPS_5Y, US_TIPS_10Y, US_TIPS_30Y,
    US_BREAKEVEN_10Y.
    """
    return _load("real_yields_daily.csv")


def load_fx():
    """
    FX spot rates from FRED. Daily frequency.
    Columns: GBPUSD, EURUSD, USDJPY.
    """
    return _load("fx_daily.csv")


def load_credit():
    """
    ICE BofA OAS credit spreads from FRED.
    Daily frequency. Columns: US_IG_OAS, US_HY_OAS, EUR_HY_OAS.
    Note: Data available from May 2023 only -- pre-2023 history behind paywall.
    UK IG OAS not available on FRED -- omitted.
    """
    return _load("credit_daily.csv")


def load_market():
    """
    Market and risk series from FRED. Daily frequency.
    Columns: VIX, BRENT, US_FED_FUNDS_DAILY, EUR_ECB_RATE.
    Note: Series extends beyond 2015 start -- filter to project sample where required.
    """
    return _load("market_daily.csv")


def load_macro():
    """
    Monthly macroeconomic series from FRED.
    Columns: US_CPI, US_CPI_CORE, US_PCE, US_PCE_CORE, US_UNEMPLOYMENT,
    US_FED_FUNDS, EUR_CPI, EUR_UNEMPLOYMENT, EUR_BUND_10Y,
    JPN_CPI, JPN_JGB_10Y, JPN_BOJ_RATE.
    Note: JPN_CPI has ~58 missing observations -- data limitation acknowledged.
    EUR_UNEMPLOYMENT incomplete post-2023 -- use load_ecb_unemployment() instead.
    """
    return _load("macro_monthly.csv")


def load_gdp():
    """
    US real GDP from FRED. Quarterly frequency.
    Columns: US_GDP_REAL.
    """
    return _load("gdp_quarterly.csv")


def load_equities():
    """
    Equity indices and commodities from yfinance. Daily frequency.
    Columns: SP500, FTSE100, EUROSTOXX, NIKKEI, GOLD, BRENT_YF.
    """
    return _load("equities_commodities_daily.csv")


# ---------------------------------------------------------------
# Bank of England Series
# ---------------------------------------------------------------

def load_boe_nominal():
    """
    UK nominal gilt spot curve from BoE Statistical Interactive Database.
    Daily frequency. Columns: 2Y, 5Y, 10Y, 30Y.
    Note: 30Y NaN prior to January 2016 -- BoE curve extended only to 25Y before that date.
    """
    return _load("boe_nominal_daily.csv")


def load_boe_real():
    """
    UK real gilt spot curve from BoE Statistical Interactive Database.
    Daily frequency. Columns: 5Y, 10Y, 30Y.
    Note: Curve begins at 5Y -- no short-maturity index-linked gilts issued.
    30Y NaN prior to January 2016.
    """
    return _load("boe_real_daily.csv")


def load_boe_inflation():
    """
    UK inflation breakeven curve from BoE Statistical Interactive Database.
    Daily frequency. Columns: 5Y, 10Y, 30Y.
    Note: Curve begins at 5Y. 30Y NaN prior to January 2016.
    """
    return _load("boe_inflation_daily.csv")


# ---------------------------------------------------------------
# ONS Series
# ---------------------------------------------------------------

def load_ons_monthly():
    """
    UK macroeconomic series from ONS. Monthly frequency.
    Columns: UK_CPI, UK_CPI_CORE, UK_CPI_SERVICES, UK_UNEMPLOYMENT.
    """
    return _load("ons_monthly.csv")


def load_ons_gdp():
    """
    UK GDP chained volume measure from ONS. Quarterly frequency.
    Columns: UK_GDP.
    """
    return _load("ons_gdp_quarterly.csv")


# ---------------------------------------------------------------
# ECB Series
# ---------------------------------------------------------------

def load_ecb_hicp():
    """
    Eurozone HICP inflation components from ECB Statistical Data Warehouse.
    Monthly frequency. Annual rate of change.
    Columns: EUR_HICP_HEADLINE, EUR_HICP_CORE, EUR_HICP_SERVICES, EUR_HICP_ENERGY.
    Use for inflation decomposition analysis -- preferred over FRED EUR_CPI series.
    """
    return _load("ecb_hicp_monthly.csv")


def load_ecb_aaa_yields():
    """
    ECB AAA-rated Eurozone government bond spot curve.
    Daily frequency. Columns: EUR_AAA_2Y, EUR_AAA_5Y, EUR_AAA_10Y, EUR_AAA_30Y.
    Note: NOT a pure German Bund series -- modelled curve fitted to AAA-rated
    Eurozone sovereigns (predominantly Germany, Netherlands, Finland).
    Use for yield curve shape and rate differential analysis only.
    Use FRED EUR_BUND_10Y as primary sovereign benchmark in relative value analysis.
    """
    return _load("ecb_aaa_yields_daily.csv")


def load_ecb_unemployment():
    """
    Eurozone unemployment rate from ECB Statistical Data Warehouse.
    Monthly frequency, seasonally adjusted, age 15-74.
    Columns: EUR_UNEMPLOYMENT_ECB.
    Preferred over FRED EUR_UNEMPLOYMENT which is incomplete post-2023.
    """
    return _load("ecb_unemployment_monthly.csv")


# ---------------------------------------------------------------
# IMF Series
# ---------------------------------------------------------------

def load_imf_output_gap():
    """
    Output gap estimates (% of potential GDP) from IMF WEO April 2026 vintage.
    Annual frequency. Columns: UK, US, Germany, Japan.
    Covers 2015-2026 including IMF forward projections from 2025.
    Note: Output gap estimates are uncertain and subject to revision between vintages.
    """
    return _load("imf_output_gap_annual.csv")
'''

# Write to src/data_loader.py with UTF-8 encoding
output_path = "../src/data_loader.py"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(loader_code)

print(f"✓ data_loader.py written to {output_path}")
print(f"  {len([l for l in loader_code.split(chr(10)) if l.strip()])} lines of code")

✓ data_loader.py written to ../src/data_loader.py
  162 lines of code


In [41]:
# ============================================================
# Verify data_loader.py imports correctly
# ============================================================

import sys
sys.path.append("..")

from src.data_loader import (
    load_yields, load_real_yields, load_fx, load_credit,
    load_market, load_macro, load_gdp, load_equities,
    load_boe_nominal, load_boe_real, load_boe_inflation,
    load_ons_monthly, load_ons_gdp,
    load_ecb_hicp, load_ecb_aaa_yields, load_ecb_unemployment,
    load_imf_output_gap
)

loaders = {
    "load_yields":             load_yields,
    "load_real_yields":        load_real_yields,
    "load_fx":                 load_fx,
    "load_credit":             load_credit,
    "load_market":             load_market,
    "load_macro":              load_macro,
    "load_gdp":                load_gdp,
    "load_equities":           load_equities,
    "load_boe_nominal":        load_boe_nominal,
    "load_boe_real":           load_boe_real,
    "load_boe_inflation":      load_boe_inflation,
    "load_ons_monthly":        load_ons_monthly,
    "load_ons_gdp":            load_ons_gdp,
    "load_ecb_hicp":           load_ecb_hicp,
    "load_ecb_aaa_yields":     load_ecb_aaa_yields,
    "load_ecb_unemployment":   load_ecb_unemployment,
    "load_imf_output_gap":     load_imf_output_gap,
}

print("Testing data_loader.py imports...\n")
all_passed = True
for name, fn in loaders.items():
    try:
        df = fn()
        print(f"  ✓ {name:<30} {df.shape}")
    except Exception as e:
        print(f"  ✗ {name:<30} FAILED — {e}")
        all_passed = False

print(f"\n{'='*55}")
print(f"{'All loaders working' if all_passed else 'Some loaders failed — check above'}")

Testing data_loader.py imports...

  ✓ load_yields                    (2995, 6)
  ✓ load_real_yields               (2957, 4)
  ✓ load_fx                        (2952, 3)
  ✓ load_credit                    (794, 3)
  ✓ load_market                    (4139, 4)
  ✓ load_macro                     (136, 12)
  ✓ load_gdp                       (45, 1)
  ✓ load_equities                  (2945, 6)
  ✓ load_boe_nominal               (2956, 4)
  ✓ load_boe_real                  (2956, 3)
  ✓ load_boe_inflation             (2956, 3)
  ✓ load_ons_monthly               (135, 4)
  ✓ load_ons_gdp                   (44, 1)
  ✓ load_ecb_hicp                  (136, 4)
  ✓ load_ecb_aaa_yields            (2889, 4)
  ✓ load_ecb_unemployment          (135, 1)
  ✓ load_imf_output_gap            (12, 4)

All loaders working
